# ARCHON - Custom Neural Network Training

Train your own LSTM neural network on your markdown documents.

## Quick Start
1. Click **Runtime** → **Change runtime type** → Select **GPU**
2. Click **Run all** (▶️) or run each cell in order
3. Download the trained model when complete
4. Upload the model to Render

In [ ]:
#@title 1. Install dependencies
!pip install torch torchdata torchvision numpy

In [ ]:
#@title 2. Clone your training data
!git clone https://github.com/thathumantorch-crypto/ARCHONAI.git /content/data
print("Data cloned!")

In [ ]:
#@title 3. Configure Training
#@markdown Adjust these hyperparameters:
SEQ_LENGTH = 100
EMBEDDING_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 3
EPOCHS = 30
BATCH_SIZE = 64
LEARNING_RATE = 0.001
DROPOUT = 0.2

print(f"Config: Embedding={EMBEDDING_DIM}, Hidden={HIDDEN_DIM}, Layers={NUM_LAYERS}, Epochs={EPOCHS}")

In [ ]:
#@title 4. Training Script
%%writefile train.py
"""
ARCHON - Custom LSTM Neural Network Training
Character-level text generation model
"""

import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ===================== DATA =====================

def load_markdown_files(data_dir="/content/data/data/docs"):
    data_path = Path(data_dir)
    texts = []
    if data_path.exists():
        for md_file in data_path.glob("*.md"):
            content = md_file.read_text(encoding="utf-8")
            texts.append(content)
    
    if not texts:
        # Fallback sample data
        texts = ["""# ARCHON Identity
ARCHON is a semi-aware artificial intelligence.
- ARCHON remembers user preferences.
- ARCHON learns from internal documents.
- ARCHON is honest about limitations.
- ARCHON is not conscious but can discuss consciousness.
- ARCHON uses neural networks for text generation.
"""]
    
    return texts

def tokenize(texts):
    combined = "\n".join(texts)
    chars = sorted(set(combined))
    char_to_idx = {c: i for i, c in enumerate(chars)}
    idx_to_char = {i: c for c, i in char_to_idx.items()}
    return combined, chars, char_to_idx, idx_to_char

def encode(text, char_to_idx):
    return [char_to_idx[c] for c in text if c in char_to_idx]

def decode(indices, idx_to_char):
    return "".join(idx_to_char.get(i, "") for i in indices)

# ===================== DATASET =====================

class TextDataset(Dataset):
    def __init__(self, text_indices, seq_length=100):
        self.text_indices = text_indices
        self.seq_length = seq_length

    def __len__(self):
        return max(0, len(self.text_indices) - self.seq_length)

    def __getitem__(self, idx):
        x = self.text_indices[idx:idx + self.seq_length]
        y = self.text_indices[idx + 1:idx + self.seq_length + 1]
        return torch.tensor(x), torch.tensor(y)

# ===================== MODEL =====================

class LSTMGenerator(nn.Module):
    def __init__(self, vocab_size, embedding_dim=256, hidden_dim=512, num_layers=3, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                          batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.vocab_size = vocab_size

    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        if hidden is None:
            output, hidden = self.lstm(embedded)
        else:
            output, hidden = self.lstm(embedded, hidden)
        logits = self.fc(output)
        return logits, hidden

    def generate(self, start_text, char_to_idx, idx_to_char, max_length=300, temperature=0.8, top_k=40):
        self.eval()
        with torch.no_grad():
            input_seq = torch.tensor([encode(start_text, char_to_idx)]).to(DEVICE)
            generated = list(start_text)
            hidden = None

            for _ in range(max_length):
                logits, hidden = self.forward(input_seq, hidden)
                logits = logits[:, -1, :] / temperature
                
                # Top-k filtering
                if top_k:
                    top_k = min(top_k, logits.size(-1))
                    topk_vals = torch.topk(logits, top_k)[0]
                    logits[logits < topk_vals[0][-1]] = float("-inf")
                
                probs = torch.softmax(logits, dim=-1)
                next_char_idx = torch.multinomial(probs, 1).item()
                next_char = idx_to_char.get(next_char_idx, "")

                if next_char == "":
                    break
                generated.append(next_char)
                input_seq = torch.tensor([[next_char_idx]]).to(DEVICE)

        return "".join(generated)

# ===================== TRAINING =====================

def train_model(model, dataloader, epochs=30, lr=0.001):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        num_batches = 0
        
        for x, y in dataloader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()

            logits, _ = model(x)
            loss = criterion(logits.view(-1, model.vocab_size), y.view(-1))

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1

        avg_loss = total_loss / max(1, num_batches)
        scheduler.step(avg_loss)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    return model

# ===================== MAIN =====================

print("Loading markdown data...")
texts = load_markdown_files()
print(f"Loaded {len(texts)} documents")

print("Tokenizing...")
combined, chars, char_to_idx, idx_to_char = tokenize(texts)
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")

print("Encoding...")
text_indices = encode(combined, char_to_idx)
print(f"Total tokens: {len(text_indices)}")

print("Creating dataset...")
dataset = TextDataset(text_indices, SEQ_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f"Batches: {len(dataloader)}")

print("Building model...")
model = LSTMGenerator(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

print("Training...")
model = train_model(model, dataloader, epochs=EPOCHS, lr=LEARNING_RATE)

print("\n" + "="*50)
print("Generating sample output...")
print("="*50)
start_seq = "# ARCHON"
generated = model.generate(start_seq, char_to_idx, idx_to_char, max_length=300)
print(f"Prompt: {start_seq}")
print(f"Generated:\n{generated}")

print("\nSaving model...")
torch.save({
    "model_state_dict": model.state_dict(),
    "char_to_idx": char_to_idx,
    "idx_to_char": idx_to_char,
    "vocab_size": vocab_size,
    "config": {
        "embedding_dim": EMBEDDING_DIM,
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "seq_length": SEQ_LENGTH,
        "dropout": DROPOUT
    }
}, "/content/archon_model.pt")
print("Model saved to /content/archon_model.pt")
print("\n" + "="*50)
print("DOWNLOAD: archon_model.pt")
print("="*50)

In [ ]:
#@title 5. Run Training
!python train.py

In [ ]:
#@title 6. Download Model
from google.colab import files
files.download("/content/archon_model.pt")